# 02. PEFT with SVD and Adaptive Allocation: PiSSA, AdaLoRA, and VeRA

- **Category:** 15_compression_peft_and_speculative_inference / 03_parameter_efficient_finetuning_peft
- **Lineage & Citations:** Meng et al. (NeurIPS 2024, *PiSSA: Principal Singular values and Singular vectors Adaptation of Large Language Models*); Zhang et al. (ICLR 2023, *Adaptive Budget Allocation for Parameter-Efficient Fine-Tuning*); Kopiczko et al. (ICLR 2024, *VeRA: Vector-based Random Matrix Adaptation*).
- **Purpose:** Implement SVD-initialized principal subspace adaptation (PiSSA), dynamic singular value importance pruning (AdaLoRA), and frozen shared random projections with vector scaling (VeRA).
- **Topic Connections:** `🔗 Upstream Prerequisites:** [PEFT LoRA Rslora Loraplus and DoRA](01_peft_lora_rslora_loraplus_and_dora.ipynb) $\to$ `🔗 Downstream Syntheses:` [PEFT Orthogonal Finetuning BOFT and Oft](03_peft_orthogonal_finetuning_boft_and_oft.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> In PiSSA, never update the residual matrix $W_{\text{res}}$ during training; modifying the residual matrix alongside the principal adapter corrupts the orthogonal subspace decomposition. In AdaLoRA, avoid aggressive early pruning before singular value importance estimates stabilize.

## Mathematical Foundations & Governing Equations

### 1. PiSSA Principal Subspace Adaptation
$$W_0 = U \Sigma V^\top, \quad B = U_{:r} \sqrt{\Sigma_{:r}}, \quad A = \sqrt{\Sigma_{:r}} V_{:r}^\top, \quad W_{\text{res}} = W_0 - B A$$
$$h = x W_{\text{res}} + (x B) A, \quad \text{Step 0: } W_{\text{res}} + B A \equiv W_0$$

### 2. AdaLoRA Dynamic SVD Allocation
$$\Delta W = P \text{diag}(\vec{\lambda} \odot \vec{m}) Q, \quad S_i = |\lambda_i \cdot \nabla_{\lambda_i} \mathcal{L}|$$

### 3. VeRA Vector-Only Parameterization
$$h = x W_0 + ((x A_{\text{shared}} \odot \vec{d}) B_{\text{shared}}) \odot \vec{b}$$
Shares $A, B$ across all layers; trains only $\vec{d} \in \mathbb{R}^r, \vec{b} \in \mathbb{R}^{d_{\text{out}}}$ ($>99\%$ storage reduction).

### Complexity
VeRA reduces per-layer parameters from $\mathcal{O}(2 d r)$ to $\mathcal{O}(r + d)$, enabling massive multi-tenant adapter colocation.

In [ ]:
import torch
import safetensors.torch
from pathlib import Path
import core

env = core.setup_env(seed=42)
CKPT_REL = Path("weights/15_compression_peft_and_speculative_inference/03_parameter_efficient_finetuning_peft/02_peft_svd_and_adaptive_pissa_adalora_vera.safetensors")
ckpt_path = next(p / CKPT_REL for p in (Path.cwd(), *Path.cwd().parents) if (p / CKPT_REL).exists())
weights = safetensors.torch.load_file(ckpt_path)
print(f"Loaded {len(weights)} golden tensors on {env.device}")

In [ ]:
def linear_anchor(x, w, b):
    return x @ w + b

def pissa_forward(x, w_res, b, a):
    return x @ w_res + (x @ b) @ a

def adalora_forward(x, w0, p, lambda_diag, q, mask):
    effective_lambda = torch.diag(lambda_diag * mask)
    delta_w = p @ effective_lambda @ q
    return x @ w0 + x @ delta_w, delta_w

def vera_forward(x, w0, a_shared, b_shared, d_vec, b_vec):
    base_out = x @ w0
    x_a = x @ a_shared
    scaled_xa = x_a * d_vec
    xa_b = scaled_xa @ b_shared
    adapter_out = xa_b * b_vec
    return base_out + adapter_out

In [ ]:
y = linear_anchor(weights["test_input"], weights["weight"], weights["bias"])
assert float(torch.max(torch.abs(y - weights["test_output"]))) < 1e-6

p_out = pissa_forward(weights["x"], weights["w_res"], weights["b_pissa"], weights["a_pissa"])
assert float(torch.max(torch.abs(p_out - weights["pissa_out"]))) < 1e-5

a_out, d_w = adalora_forward(weights["x"], weights["w0"], weights["p"], weights["lambda_diag"], weights["q"], weights["ada_mask"])
assert float(torch.max(torch.abs(a_out - weights["ada_out"]))) < 1e-5
assert float(torch.max(torch.abs(d_w - weights["delta_w"]))) < 1e-5

v_out = vera_forward(weights["x"], weights["w0"], weights["a_shared"], weights["b_shared"], weights["d_vec"], weights["b_vec"])
assert float(torch.max(torch.abs(v_out - weights["vera_out"]))) < 1e-5
print("All 4 SVD/adaptive PEFT forward modules match golden checkpoints (< 1e-5)!")

In [ ]:
from core.benchmark import benchmark_fn, format_benchmark_table

b_pissa = benchmark_fn(lambda: pissa_forward(weights["x"], weights["w_res"], weights["b_pissa"], weights["a_pissa"]), iters=100, name="PiSSA Forward (r=2)")
b_vera = benchmark_fn(lambda: vera_forward(weights["x"], weights["w0"], weights["a_shared"], weights["b_shared"], weights["d_vec"], weights["b_vec"]), iters=100, name="VeRA Forward (Shared)")
print(format_benchmark_table([b_pissa, b_vera]))

d_in, d_out, r = 4096, 4096, 8
lora_p = (d_in + d_out) * r
vera_p = r + d_out
print(f"Per-Layer Trainable Parameters: LoRA={lora_p:,} vs VeRA={vera_p:,} ({(1 - vera_p/lora_p)*100:.2f}% parameter reduction!)")

In [ ]:
# Pathological Misuse: Failing to subtract adapter from base in PiSSA duplicates energy!
w0 = weights["w0"]
b, a = weights["b_pissa"], weights["a_pissa"]
w_res = weights["w_res"]

# 1. Correct PiSSA initialization: W_res = W0 - B A
# At step 0, W_res + B A == W0 exactly
reconstructed = w_res + b @ a
diff_correct = torch.norm(reconstructed - w0).item()
assert diff_correct < 1e-5, "Correct PiSSA must exactly equal base weights at initialization!"

# 2. Flawed naive initialization: leaves W0 unmodified -> W0 + B A duplicates top singular directions!
diff_flawed = torch.norm((w0 + b @ a) - w0).item()
assert diff_flawed > 0.1, "Unsubtracted base causes severe parameter distortion!"
print(f"Verified: Correct PiSSA error ({diff_correct:.2e}) vs Unsubtracted distortion ({diff_flawed:.3f})")

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Principal Grounding (PiSSA)**: Direct adaptation of dominant singular components eliminates the random search phase of standard LoRA, speeding up convergence by up to $4\times$.
2. **Dynamic Layer Budgeting (AdaLoRA)**: Parameter sensitivity varies across depth; dynamic singular triplet pruning concentrates capacity where gradient variance is highest.
3. **Extreme Multi-Tenant Compression (VeRA)**: Freezing shared random projections allows thousands of fine-tuned models to be served concurrently from sub-megabyte vector checkpoints.

---
### Cross-Domain Generalisation Matrix
- **Signal Processing:** PiSSA mirrors Principal Component Analysis (PCA) filtering for dimension-reduced subspace tracking.
- **Compressed Sensing:** VeRA directly embodies random projection measurement matrices (Johnson-Lindenstrauss lemma).
- **Adaptive Filtering:** AdaLoRA matches variable-step LMS filters in noise-cancellation telephony.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [02_peft_svd_and_adaptive_pissa_adalora_vera.md](../../../markdown/15_compression_peft_and_speculative_inference/03_parameter_efficient_finetuning_peft/02_peft_svd_and_adaptive_pissa_adalora_vera.md)
- **PyTorch 2.x Module:** [02_peft_svd_and_adaptive_pissa_adalora_vera.py](../../../pytorch2/15_compression_peft_and_speculative_inference/03_parameter_efficient_finetuning_peft/02_peft_svd_and_adaptive_pissa_adalora_vera.py)
- **Functional JAX Module:** [02_peft_svd_and_adaptive_pissa_adalora_vera.py](../../../jax/15_compression_peft_and_speculative_inference/03_parameter_efficient_finetuning_peft/02_peft_svd_and_adaptive_pissa_adalora_vera.py)
- **Native Rust Candle Track:** [02_peft_svd_and_adaptive_pissa_adalora_vera.rs](../../../candle/15_compression_peft_and_speculative_inference/03_parameter_efficient_finetuning_peft/02_peft_svd_and_adaptive_pissa_adalora_vera.rs)
- **Hardware Kernel (CPU / WGSL):** [02_peft_svd_and_adaptive_pissa_adalora_vera.rs](../../../kernels/15_compression_peft_and_speculative_inference/03_parameter_efficient_finetuning_peft/02_peft_svd_and_adaptive_pissa_adalora_vera.rs)
- **Golden SafeTensors:** [02_peft_svd_and_adaptive_pissa_adalora_vera.safetensors](../../../weights/15_compression_peft_and_speculative_inference/03_parameter_efficient_finetuning_peft/02_peft_svd_and_adaptive_pissa_adalora_vera.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 01. PEFT LoRA Rslora Loraplus and DoRA](01_peft_lora_rslora_loraplus_and_dora.ipynb) | [Master Index](../../../README.md#15_compression_peft_and_speculative_inference) | [Next: 03. PEFT Orthogonal Finetuning BOFT and Oft](03_peft_orthogonal_finetuning_boft_and_oft.ipynb) &rarr;